<a href="https://colab.research.google.com/github/wnstj1126-debug/-/blob/main/M0_BRIDGE_2048_PREP_v1_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# M0_BRIDGE_2048_PREP_v1.1

**목적**: FEMTO 진동 2,560 samples → Edge 입력 2,048 samples 변환 방식 비교

| 항목 | 값 |
|:---|:---|
| Source Lock 정정 판정 | `SOURCE_LOCK_PASS_WITH_AUXILIARY_FILES_EXCLUDED` |
| 진동 CSV (Learning) | 7,534개 (`PASS_NATIVE_2560`) |
| 온도 보조 파일 | 850개 (`temp_*.csv`) — Bridge 입력 제외, 삭제 금지 |
| 표본 수 | Learning 6 bearings × 30 = **180개** |

## 비교 후보

| 후보 | 방식 | 출력 Fs | 지속시간 |
|:---|:---|---:|---:|
| A: CENTER_CROP_2048 | `signal[256:2304]` | 25,600 Hz | 0.08 s |
| B: DURATION_PRESERVING_2048 | `resample_poly(up=4, down=5)` | 20,480 Hz | 0.1 s |

**추천**: B — 0.1초 유지, net downsampling, 저율 upsampling 아님

## Candidate B 통과 기준

- 0~6kHz energy relative error median ≤ 0.05
- 0~6kHz energy relative error p95 ≤ 0.10
- dominant-frequency absolute error median ≤ 50 Hz
- RMS relative error median ≤ 0.05
- bearing/phase별 conversion failure 0건
- 기준 완화 금지

## 실행 순서

1. **셀 01** — 단위 검증 (드라이브 마운트 불필요)
2. **셀 02** — Source Classification 정정 + 분류 검증
3. **셀 03** — 180개 표본 선택
4. **셀 04** — Bridge A/B 변환 + 품질 평가
5. **셀 05** — 최종 판정 + 요약 생성
6. **셀 06** — 결과 확인 (독립 실행 가능)

## 절대 원칙

- 원본 CSV 수정·삭제·이동 **금지** (temp 파일 포함)
- TEST/FULL_TEST 데이터는 Bridge 방식 선택에 **사용 금지**
- 통과 기준 자동 완화 **금지**
- 실패 파일을 몰래 제외하고 PASS 처리 **금지**

In [10]:
# ================================================================
# 셀 01 — 단위 검증 (드라이브 마운트 불필요)
# ================================================================

import re
import numpy as np
import scipy.signal

# ── 기본 상수 ─────────────────────────────────────────────────
assert int(25600 * 0.1) == 2560
assert int(20480 * 0.1) == 2048
# resample_poly(up=4, down=5): 2560 * 4 / 5 = 2048
assert int(2560 * 4 / 5) == 2048
print("[Unit] 상수 assert 통과 ✅")

# ── CENTER_CROP_2048 ─────────────────────────────────────────
_sig = np.arange(2560, dtype=float)
_crop = _sig[256:2304]
assert len(_crop) == 2048, f"CENTER_CROP 길이 오류: {len(_crop)}"
print("[Unit] CENTER_CROP_2048 통과 ✅")

# ── DURATION_PRESERVING_2048 ─────────────────────────────────
_rsig = np.ones(2560)
_rout = scipy.signal.resample_poly(_rsig, up=4, down=5, padtype="line")
assert len(_rout) == 2048, f"resample_poly 길이 오류: {len(_rout)}"
print("[Unit] DURATION_PRESERVING_2048 통과 ✅")

# ── temp 파일명 패턴 ─────────────────────────────────────────
_temp_pat = re.compile(r"^temp_\d+\.csv$", re.IGNORECASE)
assert _temp_pat.match("temp_001.csv")
assert _temp_pat.match("temp_12345.csv")
assert not _temp_pat.match("acc_001.csv")
assert not _temp_pat.match("temp_abc.csv")
print("[Unit] temp 패턴 통과 ✅")

# ── natural sort ─────────────────────────────────────────────
def _nk(s):
    return [int(t) if t.isdigit() else t.lower()
            for t in re.split(r"(\d+)", str(s))]

assert sorted(["acc_10.csv","acc_2.csv","acc_1.csv"], key=_nk) == \
       ["acc_1.csv","acc_2.csv","acc_10.csv"]
print("[Unit] natural sort 통과 ✅")

# ── relative error guard (분모 < 1e-12) ─────────────────────
def _rel_err(a, b):
    if abs(b) < 1e-12:
        return None  # LOW_ENERGY_UNDEFINED
    return abs(a - b) / abs(b)

assert _rel_err(1.0, 0.0) is None
assert abs(_rel_err(1.1, 1.0) - 0.1) < 1e-9
print("[Unit] relative error guard 통과 ✅")

# ── 표본 선택 균등 분포 ─────────────────────────────────────
def _linspace_indices(total, n=10):
    return [int(round(i)) for i in np.linspace(0, total-1, n)]

_idx = _linspace_indices(100, 10)
assert len(_idx) == 10
assert len(set(_idx)) == 10, f"중복 index: {_idx}"
print("[Unit] 표본 선택 통과 ✅")

print("\n[Unit] 모든 단위 검증 통과 ✅")

[Unit] 상수 assert 통과 ✅
[Unit] CENTER_CROP_2048 통과 ✅
[Unit] DURATION_PRESERVING_2048 통과 ✅
[Unit] temp 패턴 통과 ✅
[Unit] natural sort 통과 ✅
[Unit] relative error guard 통과 ✅
[Unit] 표본 선택 통과 ✅

[Unit] 모든 단위 검증 통과 ✅


In [11]:
# ================================================================
# 셀 02 — Source Classification 정정 + 분류 검증
# ================================================================

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import hashlib, json, math, os, re, traceback
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.signal


# ── 고정 설정 ─────────────────────────────────────────────────
VERSION = "M0_BRIDGE_2048_PREP_v1.1"

SOURCE_FS       = 25600
SOURCE_SAMPLES  = 2560
SOURCE_DUR      = 0.1
COL_H           = 4
OUTPUT_SAMPLES  = 2048
OUTPUT_FS       = 20480
RESAMPLE_UP     = 4
RESAMPLE_DOWN   = 5
CROP_START      = 256
CROP_END        = 2304
COMMON_BAND_HZ  = (0, 6000)

LEARNING_VIB_EXPECTED   = 7534
LEARNING_TEMP_EXPECTED  = 850
LEARNING_TOTAL_EXPECTED = 8384
SAMPLES_PER_BEARING     = 30     # 10 × 3 phases
PHASES                  = ["EARLY", "MIDDLE", "LATE"]
SAMPLES_PER_PHASE       = 10
TOTAL_SAMPLES           = 180

TEMP_PAT = re.compile(r"^temp_\d+\.csv$", re.IGNORECASE)

LEARNING_BEARINGS = [
    "Bearing1_1", "Bearing1_2",
    "Bearing2_1", "Bearing2_2",
    "Bearing3_1", "Bearing3_2",
]

BRIDGE_QUALITY_CRITERIA = {
    "energy_0_6khz_relerr_median": 0.05,
    "energy_0_6khz_relerr_p95":    0.10,
    "dominant_freq_abserr_median": 50.0,
    "rms_relerr_median":           0.05,
}

PROJECT_ROOT_CANDIDATES = [
    Path("/content/drive/MyDrive/Colab Notebooks/field_iis3dwb"),
    Path("/content/drive/My Drive/Colab Notebooks/field_iis3dwb"),
]

PROJECT_ROOT = None
for _c in PROJECT_ROOT_CANDIDATES:
    if _c.exists():
        PROJECT_ROOT = _c
        break
if PROJECT_ROOT is None:
    raise FileNotFoundError("PROJECT_ROOT_NOT_FOUND")

RUN_ID     = datetime.now().strftime("%Y%m%d_%H%M%S_bridge_prep_v1_1")
OUTPUT_DIR = PROJECT_ROOT / "bridge_outputs" / RUN_ID
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)

print(f"PROJECT_ROOT : {PROJECT_ROOT}")
print(f"OUTPUT_DIR   : {OUTPUT_DIR}")


# ── 공통 함수 ─────────────────────────────────────────────────
def write_json(path, obj):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with tmp.open("w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2, default=str)
    os.replace(tmp, path)

def natural_key(s):
    return [int(t) if t.isdigit() else t.lower()
            for t in re.split(r"(\d+)", str(s))]

def sha256_file(path, chunk=1<<20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            c = f.read(chunk)
            if not c: break
            h.update(c)
    return h.hexdigest()

def rel_err(a, b):
    if abs(b) < 1e-12:
        return None
    return abs(a - b) / abs(b)


# ── 기존 v1.1 / v1.2 결과 탐색 ───────────────────────────────
def find_latest_dir(pattern):
    dirs = sorted(
        [p for p in (PROJECT_ROOT / "bridge_outputs").glob(pattern)
         if p.is_dir()],
        key=lambda p: p.stat().st_mtime_ns, reverse=True,
    )
    return dirs[0] if dirs else None

V1_DIR    = find_latest_dir("*_source_lock_v1_1")
V1_2_DIR  = find_latest_dir("*_source_lock_schema_v1_2")

if V1_DIR is None:
    raise FileNotFoundError("SOURCE_LOCK_V1_1_NOT_FOUND")
if V1_2_DIR is None:
    raise FileNotFoundError("SOURCE_LOCK_SCHEMA_V1_2_NOT_FOUND")

SCHEMA_RESULT_DIR = V1_2_DIR
print(f"V1_DIR    : {V1_DIR}")
print(f"V1_2_DIR  : {V1_2_DIR}")

# ── 기존 결과 로드 ────────────────────────────────────────────
manifest    = pd.read_csv(V1_DIR / "femto_csv_source_manifest.csv")
schema      = pd.read_csv(SCHEMA_RESULT_DIR / "learning_schema_full.csv")
v1_summary  = json.loads((V1_DIR / "SOURCE_LOCK_SUMMARY.json").read_text("utf-8"))
audit_gate  = json.loads((V1_DIR / "audit_gate_check.json").read_text("utf-8"))
immut_base  = pd.read_csv(V1_DIR / "source_file_immutability_check.csv")

print(f"manifest rows : {len(manifest):,}")
print(f"schema rows   : {len(schema):,}")


# ── Source 분류 ───────────────────────────────────────────────
schema["is_temp_file"] = (
    schema["csv_file_name"]
    .astype(str)
    .str.match(r"^temp_\d+\.csv$", case=False, na=False)
)

schema["is_vibration"] = (
    schema["schema_status"].eq("PASS_NATIVE_2560")
    & schema["data_row_count"].eq(2560)
    & schema["column_count"].ge(5)
    & ~schema["is_temp_file"]
)

schema["is_aux_temperature"] = (
    schema["is_temp_file"]
    & schema["schema_status"].eq("FAIL_UNRESOLVED_SCHEMA")
)

schema["source_class"] = "UNCLASSIFIED"
schema.loc[schema["is_vibration"],         "source_class"] = "VIBRATION"
schema.loc[schema["is_aux_temperature"],   "source_class"] = "AUXILIARY_TEMPERATURE"


# ── 분류 검증 (5개 조건) ─────────────────────────────────────
classification_checks = {
    "total_learning_csv":   len(schema) == LEARNING_TOTAL_EXPECTED,
    "vibration_count":      int(schema["is_vibration"].sum()) == LEARNING_VIB_EXPECTED,
    "temperature_count":    int(schema["is_aux_temperature"].sum()) == LEARNING_TEMP_EXPECTED,
    "temp_in_vibration":    int((schema["is_temp_file"] & schema["is_vibration"]).sum()) == 0,
    "non_temp_failure":     int((~schema["is_temp_file"] & ~schema["is_vibration"]).sum()) == 0,
}

classification_pass = all(classification_checks.values())

SOURCE_LOCK_STATUS = (
    "SOURCE_LOCK_PASS_WITH_AUXILIARY_FILES_EXCLUDED"
    if classification_pass
    else "SOURCE_CLASSIFICATION_REVIEW_REQUIRED"
)

print("\n=== Source 분류 검증 ===")
for k, v in classification_checks.items():
    icon = "✅" if v else "❌"
    print(f"  {icon} {k}: {v}")
print(f"  classification_pass = {classification_pass}")
print(f"  SOURCE_LOCK_STATUS  = {SOURCE_LOCK_STATUS}")

if not classification_pass:
    print("\n⚠️  분류 검증 실패 — 이후 셀 계속 실행하지 마십시오")


# ── Source classification 결과 저장 ──────────────────────────
schema.to_csv(
    OUTPUT_DIR / "source_classification_manifest.csv",
    index=False, encoding="utf-8-sig"
)

aux_temp = schema[schema["is_aux_temperature"]].copy()
aux_temp.to_csv(
    OUTPUT_DIR / "auxiliary_temperature_inventory.csv",
    index=False, encoding="utf-8-sig"
)

closure = {
    "version":                            VERSION,
    "created_at":                         datetime.now().isoformat(timespec="seconds"),
    "source_lock_status":                 SOURCE_LOCK_STATUS,
    "recommended_next_action":            "READY_FOR_M0_BRIDGE_2048_PREP" if classification_pass else "REVIEW_CLASSIFICATION",
    "learning_total_csv":                 len(schema),
    "learning_vibration_csv":             int(schema["is_vibration"].sum()),
    "learning_auxiliary_temperature_csv": int(schema["is_aux_temperature"].sum()),
    "classification_checks":              classification_checks,
    "classification_pass":                classification_pass,
    "temp_files_deleted":                 False,
    "temp_files_moved":                   False,
    "original_files_modified":            False,
}
write_json(OUTPUT_DIR / "source_classification_closure.json", closure)
print(f"\n저장: source_classification_closure.json")
print(f"      auxiliary_temperature_inventory.csv ({len(aux_temp)}개)")

if not classification_pass:
    raise RuntimeError("SOURCE_CLASSIFICATION_REVIEW_REQUIRED")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PROJECT_ROOT : /content/drive/MyDrive/Colab Notebooks/field_iis3dwb
OUTPUT_DIR   : /content/drive/MyDrive/Colab Notebooks/field_iis3dwb/bridge_outputs/20260725_192744_bridge_prep_v1_1
V1_DIR    : /content/drive/MyDrive/Colab Notebooks/field_iis3dwb/bridge_outputs/20260725_150815_source_lock_v1_1
V1_2_DIR  : /content/drive/MyDrive/Colab Notebooks/field_iis3dwb/bridge_outputs/20260725_155856_source_lock_schema_v1_2
manifest rows : 43,369
schema rows   : 8,384

=== Source 분류 검증 ===
  ✅ total_learning_csv: True
  ✅ vibration_count: True
  ✅ temperature_count: True
  ✅ temp_in_vibration: True
  ✅ non_temp_failure: True
  classification_pass = True
  SOURCE_LOCK_STATUS  = SOURCE_LOCK_PASS_WITH_AUXILIARY_FILES_EXCLUDED

저장: source_classification_closure.json
      auxiliary_temperature_inventory.csv (850개)


In [12]:
# ================================================================
# 셀 03 — 180개 표본 선택
# ================================================================

# ── 진동 CSV 필터링 ──────────────────────────────────────────
vibration_df = schema[
    schema["is_vibration"] == True
].copy()

print(f"진동 CSV (Learning): {len(vibration_df):,}")


def linspace_indices_dedup(total, n, phase_range):
    """
    전체 total개에서 phase_range=(start_ratio, end_ratio)에 해당하는
    index 구간에서 n개를 균등 선택한다.
    중복 발생 시 인접한 미선정 index로 대체한다.
    """
    start_idx = int(round(total * phase_range[0]))
    end_idx   = min(int(round(total * phase_range[1])) - 1, total - 1)
    span      = end_idx - start_idx + 1

    if span < n:
        # 구간이 너무 짧으면 가능한 만큼만
        return list(range(start_idx, start_idx + span))

    raw = [int(round(i)) for i in np.linspace(start_idx, end_idx, n)]

    # 중복 제거 후 인접 미선정으로 대체
    seen = set()
    result = []
    for idx in raw:
        if idx not in seen:
            seen.add(idx)
            result.append(idx)
        else:
            # 인접한 미선정 index 탐색
            for delta in range(1, span):
                for candidate in [idx + delta, idx - delta]:
                    if (start_idx <= candidate <= end_idx
                            and candidate not in seen):
                        seen.add(candidate)
                        result.append(candidate)
                        break
                else:
                    continue
                break

    return sorted(result[:n])


PHASE_RANGES = {
    "EARLY":  (0.0,      1/3),
    "MIDDLE": (1/3,      2/3),
    "LATE":   (2/3,      1.0),
}

sample_rows = []

for bearing_id in LEARNING_BEARINGS:
    bear_df = (
        vibration_df[vibration_df["logical_bearing_id"] == bearing_id]
        .sort_values("sequence_index")
        .reset_index(drop=True)
    )
    total = len(bear_df)

    if total == 0:
        print(f"  ⚠️  {bearing_id}: 진동 CSV 없음")
        continue

    for phase, phase_range in PHASE_RANGES.items():
        indices = linspace_indices_dedup(total, SAMPLES_PER_PHASE, phase_range)
        for i, idx in enumerate(indices):
            row = bear_df.iloc[idx]
            sample_rows.append({
                "sample_id":           f"{bearing_id}_{phase}_{i:02d}",
                "bearing_id":          bearing_id,
                "phase":               phase,
                "bearing_index":       idx,
                "sequence_index":      int(row["sequence_index"]),
                "record_uid":          row["record_uid"],
                "csv_file_name":       row["csv_file_name"],
                "absolute_path":       row["absolute_path"],
                "file_size_bytes":     int(row["file_size_bytes"]),
                "mtime_ns":            int(row["mtime_ns"]),
                "schema_status":       row["schema_status"],
                "is_temp_file":        bool(row["is_temp_file"]),
            })

SAMPLE_DF = pd.DataFrame(sample_rows)

# ── 표본 검증 ─────────────────────────────────────────────────
sample_checks = {
    "total_180":           len(SAMPLE_DF) == TOTAL_SAMPLES,
    "per_bearing_30":      all(
        len(SAMPLE_DF[SAMPLE_DF["bearing_id"]==b]) == SAMPLES_PER_BEARING
        for b in LEARNING_BEARINGS
    ),
    "per_phase_10":        all(
        len(SAMPLE_DF[
            (SAMPLE_DF["bearing_id"]==b) & (SAMPLE_DF["phase"]==p)
        ]) == SAMPLES_PER_PHASE
        for b in LEARNING_BEARINGS for p in PHASES
    ),
    "no_dup_path":         len(SAMPLE_DF["absolute_path"].unique()) == TOTAL_SAMPLES,
    "no_temp_file":        int(SAMPLE_DF["is_temp_file"].sum()) == 0,
}

print("\n=== 표본 선택 검증 ===")
for k, v in sample_checks.items():
    icon = "✅" if v else "❌"
    print(f"  {icon} {k}: {v}")

if not all(sample_checks.values()):
    raise RuntimeError(f"SAMPLE_SELECTION_FAILED: {sample_checks}")

SAMPLE_DF.to_csv(
    OUTPUT_DIR / "bridge_sample_manifest.csv",
    index=False, encoding="utf-8-sig"
)
print(f"\n저장: bridge_sample_manifest.csv ({len(SAMPLE_DF)}개)")
display(SAMPLE_DF.groupby(["bearing_id","phase"]).size().reset_index(name="count"))

진동 CSV (Learning): 7,534

=== 표본 선택 검증 ===
  ✅ total_180: True
  ✅ per_bearing_30: True
  ✅ per_phase_10: True
  ✅ no_dup_path: True
  ✅ no_temp_file: True

저장: bridge_sample_manifest.csv (180개)


,bearing_id,phase,count
0,Bearing1_1,EARLY,10
1,Bearing1_1,LATE,10
2,Bearing1_1,MIDDLE,10
3,Bearing1_2,EARLY,10
4,Bearing1_2,LATE,10
5,Bearing1_2,MIDDLE,10
6,Bearing2_1,EARLY,10
7,Bearing2_1,LATE,10
8,Bearing2_1,MIDDLE,10
9,Bearing2_2,EARLY,10


In [13]:
# ================================================================
# 셀 04 — Bridge A/B 변환 + 품질 평가
# ================================================================

from scipy.signal import periodogram, resample_poly
from scipy.stats  import kurtosis, skew


# ── 원본 파일 사전 해시 (180개 + 기준 파일) ──────────────────
REFERENCE_NAMES = [
    "BASELINE_M0_frozen.json",
    "m0_baseline_result.csv",
]
REF_NB_NAMES = [
    "M0_FEMTO_Baseline_v1_baseline고정.ipynb",
    "M0_FEMTO_Baseline_v1_baseline.ipynb",
    "M0_FEMTO_Baseline_v1.ipynb",
]

immut_records = {}

def _snap(path, label):
    p = Path(path)
    if not p.exists():
        return
    st = p.stat()
    immut_records[str(p.resolve())] = {
        "label": label,
        "size_before": int(st.st_size),
        "mtime_ns_before": int(st.st_mtime_ns),
        "sha256_before": sha256_file(p),
    }

for nm in REFERENCE_NAMES:
    hits = sorted(PROJECT_ROOT.rglob(nm), key=lambda p: p.stat().st_mtime, reverse=True)
    if hits: _snap(hits[0], "M0_REFERENCE")

for nm in REF_NB_NAMES:
    hits = sorted(PROJECT_ROOT.rglob(nm), key=lambda p: p.stat().st_mtime, reverse=True)
    if hits:
        _snap(hits[0], "M0_NOTEBOOK")
        break

for _, row in SAMPLE_DF.iterrows():
    _snap(row["absolute_path"], "SAMPLE_CSV")

print(f"[IMMUT] 사전 해시 완료: {len(immut_records)}개")


# ── PSD 계산 공통 함수 ────────────────────────────────────────
def compute_psd(signal, fs, window="hann", detrend="constant", scaling="density"):
    freqs, psd = periodogram(
        signal, fs=fs,
        window=window, detrend=detrend, scaling=scaling,
    )
    return freqs, psd


def band_energy(freqs, psd, flo, fhi):
    mask = (freqs >= flo) & (freqs <= fhi)
    if not mask.any():
        return 0.0
    return float(np.trapz(psd[mask], freqs[mask]))


def dominant_freq(freqs, psd, flo, fhi):
    mask = (freqs >= flo) & (freqs <= fhi)
    if not mask.any():
        return None
    idx = np.argmax(psd[mask])
    return float(freqs[mask][idx])


def spectral_centroid(freqs, psd, flo, fhi):
    mask = (freqs >= flo) & (freqs <= fhi)
    if not mask.any():
        return None
    total = np.sum(psd[mask])
    if total < 1e-30:
        return None
    return float(np.sum(freqs[mask] * psd[mask]) / total)


def time_metrics(sig):
    rms    = float(np.sqrt(np.mean(sig**2)))
    peak   = float(np.max(np.abs(sig)))
    crest  = peak / rms if rms > 1e-12 else None
    return {
        "mean": float(np.mean(sig)),
        "rms": rms,
        "std": float(np.std(sig)),
        "peak": peak,
        "peak_to_peak": float(np.ptp(sig)),
        "kurtosis": float(kurtosis(sig)),
        "skewness": float(skew(sig)),
        "crest_factor": crest,
    }


def freq_metrics(freqs, psd):
    flo, fhi = COMMON_BAND_HZ
    e02 = band_energy(freqs, psd, 0,    2000)
    e24 = band_energy(freqs, psd, 2000, 4000)
    e46 = band_energy(freqs, psd, 4000, 6000)
    e06 = band_energy(freqs, psd, flo,  fhi)
    return {
        "e_0_2khz": e02,
        "e_2_4khz": e24,
        "e_4_6khz": e46,
        "e_0_6khz": e06,
        "ratio_0_2": e02/e06 if e06 > 1e-30 else None,
        "ratio_2_4": e24/e06 if e06 > 1e-30 else None,
        "ratio_4_6": e46/e06 if e06 > 1e-30 else None,
        "dominant_freq_0_6khz": dominant_freq(freqs, psd, flo, fhi),
        "spectral_centroid_0_6khz": spectral_centroid(freqs, psd, flo, fhi),
    }


# ── Bridge 평가 메인 루프 ─────────────────────────────────────
rows_A, rows_B, fail_rows = [], [], []
TOTAL = len(SAMPLE_DF)

for idx_s, srow in SAMPLE_DF.iterrows():
    fpath  = Path(srow["absolute_path"])
    sid    = srow["sample_id"]

    base_info = {
        "sample_id":      sid,
        "bearing_id":     srow["bearing_id"],
        "phase":          srow["phase"],
        "csv_file_name":  srow["csv_file_name"],
        "sequence_index": srow["sequence_index"],
    }

    try:
        df_src = pd.read_csv(fpath, header=None)
        src    = pd.to_numeric(df_src.iloc[:, COL_H], errors="coerce").to_numpy(dtype=float)
        assert len(src) == SOURCE_SAMPLES, f"source len={len(src)}"
        assert np.isfinite(src).all(), "source has NaN/Inf"

        # ── Candidate A: CENTER_CROP_2048 ────────────────────
        sig_A      = src[CROP_START:CROP_END]
        assert len(sig_A) == OUTPUT_SAMPLES
        src_crop   = src[CROP_START:CROP_END]   # 비교 기준: 동일 구간

        tm_A       = time_metrics(sig_A)
        f_A, p_A   = compute_psd(sig_A,   SOURCE_FS)
        f_src, p_sc = compute_psd(src_crop, SOURCE_FS)
        fm_A       = freq_metrics(f_A, p_A)
        fm_sc      = freq_metrics(f_src, p_sc)

        rms_re_A   = rel_err(tm_A["rms"], float(np.sqrt(np.mean(src_crop**2))))
        e_re_A     = rel_err(fm_A["e_0_6khz"], fm_sc["e_0_6khz"])
        df_re_A    = abs(fm_A["dominant_freq_0_6khz"] - fm_sc["dominant_freq_0_6khz"]) \
                     if None not in [fm_A["dominant_freq_0_6khz"], fm_sc["dominant_freq_0_6khz"]] else None

        row_A = {
            **base_info,
            "candidate": "CENTER_CROP_2048",
            "output_samples": len(sig_A),
            "output_fs_hz": SOURCE_FS,
            "output_duration_sec": len(sig_A)/SOURCE_FS,
            "source_coverage_pct": 80.0,
            "nan_count": int(np.isnan(sig_A).sum()),
            "inf_count": int(np.isinf(sig_A).sum()),
            **{f"tm_{k}": v for k, v in tm_A.items()},
            **{f"fm_{k}": v for k, v in fm_A.items()},
            "rms_relerr":             rms_re_A,
            "energy_0_6khz_relerr":   e_re_A,
            "dominant_freq_abserr":   df_re_A,
            "metric_status":          "OK" if None not in [rms_re_A, e_re_A] else "LOW_ENERGY_UNDEFINED",
            "status": "PASS",
        }
        rows_A.append(row_A)

        # ── Candidate B: DURATION_PRESERVING_2048 ────────────
        sig_B = resample_poly(src, up=RESAMPLE_UP, down=RESAMPLE_DOWN, padtype="line")

        if len(sig_B) != OUTPUT_SAMPLES:
            fail_rows.append({**base_info, "candidate": "B",
                              "fail_reason": f"output_len={len(sig_B)}"})
            rows_B.append({**base_info, "candidate": "DURATION_PRESERVING_2048",
                           "status": "FAIL_LENGTH", "output_samples": len(sig_B)})
            continue

        assert np.isfinite(sig_B).all(), "B has NaN/Inf"

        tm_B       = time_metrics(sig_B)
        f_B, p_B   = compute_psd(sig_B, OUTPUT_FS)
        f_orig, p_orig = compute_psd(src, SOURCE_FS)
        fm_B       = freq_metrics(f_B, p_B)
        fm_orig    = freq_metrics(f_orig, p_orig)

        tm_src     = time_metrics(src)
        rms_re_B   = rel_err(tm_B["rms"],    tm_src["rms"])
        e_re_B     = rel_err(fm_B["e_0_6khz"], fm_orig["e_0_6khz"])
        df_re_B    = abs(fm_B["dominant_freq_0_6khz"] - fm_orig["dominant_freq_0_6khz"]) \
                     if None not in [fm_B["dominant_freq_0_6khz"], fm_orig["dominant_freq_0_6khz"]] else None
        sc_re_B    = abs(fm_B["spectral_centroid_0_6khz"] - fm_orig["spectral_centroid_0_6khz"]) \
                     if None not in [fm_B["spectral_centroid_0_6khz"], fm_orig["spectral_centroid_0_6khz"]] else None

        # band-ratio absolute errors
        br_errs = {
            "band_ratio_0_2_abserr": abs(fm_B["ratio_0_2"] - fm_orig["ratio_0_2"])
                                     if None not in [fm_B["ratio_0_2"], fm_orig["ratio_0_2"]] else None,
            "band_ratio_2_4_abserr": abs(fm_B["ratio_2_4"] - fm_orig["ratio_2_4"])
                                     if None not in [fm_B["ratio_2_4"], fm_orig["ratio_2_4"]] else None,
            "band_ratio_4_6_abserr": abs(fm_B["ratio_4_6"] - fm_orig["ratio_4_6"])
                                     if None not in [fm_B["ratio_4_6"], fm_orig["ratio_4_6"]] else None,
        }

        row_B = {
            **base_info,
            "candidate": "DURATION_PRESERVING_2048",
            "output_samples": len(sig_B),
            "output_fs_hz": OUTPUT_FS,
            "output_duration_sec": len(sig_B)/OUTPUT_FS,
            "nan_count": int(np.isnan(sig_B).sum()),
            "inf_count": int(np.isinf(sig_B).sum()),
            "no_low_rate_upsampling": True,
            "output_info_len_not_increased": True,
            **{f"tm_{k}": v for k, v in tm_B.items()},
            **{f"fm_{k}": v for k, v in fm_B.items()},
            "rms_relerr":             rms_re_B,
            "energy_0_6khz_relerr":   e_re_B,
            "dominant_freq_abserr":   df_re_B,
            "spectral_centroid_abserr": sc_re_B,
            **br_errs,
            "metric_status": "OK" if None not in [rms_re_B, e_re_B] else "LOW_ENERGY_UNDEFINED",
            "status": "PASS",
        }
        rows_B.append(row_B)

    except Exception as e:
        fail_rows.append({**base_info, "candidate": "BOTH", "fail_reason": repr(e)})
        rows_A.append({**base_info, "candidate": "CENTER_CROP_2048",
                       "status": "ERROR", "error_message": repr(e)})
        rows_B.append({**base_info, "candidate": "DURATION_PRESERVING_2048",
                       "status": "ERROR", "error_message": repr(e)})

    if (idx_s + 1) % 30 == 0:
        print(f"  [{idx_s+1}/{TOTAL}] 처리 중...")

CAND_A_DF = pd.DataFrame(rows_A)
CAND_B_DF = pd.DataFrame(rows_B)
FAIL_DF   = pd.DataFrame(fail_rows)

CAND_A_DF.to_csv(OUTPUT_DIR / "bridge_candidate_A_metrics.csv", index=False, encoding="utf-8-sig")
CAND_B_DF.to_csv(OUTPUT_DIR / "bridge_candidate_B_metrics.csv", index=False, encoding="utf-8-sig")
FAIL_DF.to_csv(  OUTPUT_DIR / "bridge_failures.csv",            index=False, encoding="utf-8-sig")

print(f"\n[EVAL] Candidate A: {len(CAND_A_DF)}개  Candidate B: {len(CAND_B_DF)}개  Failures: {len(FAIL_DF)}개")

[IMMUT] 사전 해시 완료: 183개


/tmp/ipykernel_2861/197287997.py:63: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(psd[mask], freqs[mask]))
/tmp/ipykernel_2861/197287997.py:63: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(psd[mask], freqs[mask]))
/tmp/ipykernel_2861/197287997.py:63: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(psd[mask], freqs[mask]))
/tmp/ipykernel_2861/197287997.py:63: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(psd[mask], freqs[mask]))
/tmp/ipykernel_2861/197287997.py:63: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one 

  [30/180] 처리 중...


/tmp/ipykernel_2861/197287997.py:63: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(psd[mask], freqs[mask]))
/tmp/ipykernel_2861/197287997.py:63: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(psd[mask], freqs[mask]))
/tmp/ipykernel_2861/197287997.py:63: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(psd[mask], freqs[mask]))
/tmp/ipykernel_2861/197287997.py:63: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(psd[mask], freqs[mask]))
/tmp/ipykernel_2861/197287997.py:63: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one 

  [60/180] 처리 중...


...

/tmp/ipykernel_2861/197287997.py:63: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(psd[mask], freqs[mask]))
/tmp/ipykernel_2861/197287997.py:63: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(psd[mask], freqs[mask]))
/tmp/ipykernel_2861/197287997.py:63: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(psd[mask], freqs[mask]))
/tmp/ipykernel_2861/197287997.py:63: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(psd[mask], freqs[mask]))
/tmp/ipykernel_2861/197287997.py:63: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one 

  [150/180] 처리 중...


/tmp/ipykernel_2861/197287997.py:63: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(psd[mask], freqs[mask]))
/tmp/ipykernel_2861/197287997.py:63: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(psd[mask], freqs[mask]))
/tmp/ipykernel_2861/197287997.py:63: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(psd[mask], freqs[mask]))
/tmp/ipykernel_2861/197287997.py:63: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(psd[mask], freqs[mask]))
/tmp/ipykernel_2861/197287997.py:63: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one 

  [180/180] 처리 중...

[EVAL] Candidate A: 180개  Candidate B: 180개  Failures: 0개


/tmp/ipykernel_2861/197287997.py:63: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(psd[mask], freqs[mask]))
/tmp/ipykernel_2861/197287997.py:63: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(psd[mask], freqs[mask]))
/tmp/ipykernel_2861/197287997.py:63: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(psd[mask], freqs[mask]))
/tmp/ipykernel_2861/197287997.py:63: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(psd[mask], freqs[mask]))
/tmp/ipykernel_2861/197287997.py:63: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one 

In [14]:
# ================================================================
# 셀 05 — 최종 판정 + 요약 생성
# ================================================================

# ── Candidate B 구조 기준 ────────────────────────────────────
b_ok = CAND_B_DF[CAND_B_DF["status"] == "PASS"].copy()

struct_pass = {
    "all_180_output_2048":      len(b_ok) == TOTAL_SAMPLES and
                                (b_ok["output_samples"] == OUTPUT_SAMPLES).all(),
    "no_nan_inf":               ((b_ok["nan_count"] + b_ok["inf_count"]) == 0).all() if len(b_ok)>0 else False,
    "duration_0_1_sec":         ((b_ok["output_duration_sec"] - 0.1).abs() < 1e-9).all() if len(b_ok)>0 else False,
    "effective_fs_20480":       (b_ok["output_fs_hz"] == OUTPUT_FS).all() if len(b_ok)>0 else False,
    "no_low_rate_upsampling":   bool(b_ok["no_low_rate_upsampling"].all()) if len(b_ok)>0 else False,
    "info_len_not_increased":   bool(b_ok["output_info_len_not_increased"].all()) if len(b_ok)>0 else False,
    "bearing_phase_failure_0":  len(FAIL_DF) == 0,
}

# ── Candidate B 품질 기준 ────────────────────────────────────
def safe_median(series):
    v = series.dropna()
    return float(v.median()) if len(v) > 0 else None

def safe_pct(series, q):
    v = series.dropna()
    return float(np.percentile(v, q)) if len(v) > 0 else None

b_metrics = {
    "energy_0_6khz_relerr_median": safe_median(b_ok.get("energy_0_6khz_relerr", pd.Series(dtype=float))),
    "energy_0_6khz_relerr_p95":    safe_pct(b_ok.get("energy_0_6khz_relerr", pd.Series(dtype=float)), 95),
    "dominant_freq_abserr_median": safe_median(b_ok.get("dominant_freq_abserr", pd.Series(dtype=float))),
    "rms_relerr_median":           safe_median(b_ok.get("rms_relerr", pd.Series(dtype=float))),
}

quality_pass = {
    k: (v is not None and v <= BRIDGE_QUALITY_CRITERIA[k])
    for k, v in b_metrics.items()
}

B_STRUCT_PASS  = all(struct_pass.values())
B_QUALITY_PASS = all(quality_pass.values())
B_TOTAL_PASS   = B_STRUCT_PASS and B_QUALITY_PASS

print("=== Candidate B 구조 기준 ===")
for k, v in struct_pass.items():
    print(f"  {'✅' if v else '❌'} {k}: {v}")

print("\n=== Candidate B 품질 기준 ===")
for k, v in quality_pass.items():
    mval = b_metrics[k]
    thr  = BRIDGE_QUALITY_CRITERIA[k]
    print(f"  {'✅' if v else '❌'} {k}: {mval:.4f} <= {thr}  → {v}")

# ── bridge_status 결정 ───────────────────────────────────────
if not classification_pass:
    BRIDGE_STATUS = "SOURCE_CLASSIFICATION_REVIEW_REQUIRED"
    NEXT_ACTION   = "REVIEW_CLASSIFICATION"
elif B_TOTAL_PASS:
    BRIDGE_STATUS = "BRIDGE_SPEC_READY"
    NEXT_ACTION   = "READY_FOR_M0_BRIDGE_2048_APPLY"
else:
    BRIDGE_STATUS = "BRIDGE_DESIGN_REVIEW_REQUIRED"
    NEXT_ACTION   = "REVIEW_BRIDGE_DESIGN"

print(f"\nBRIDGE_STATUS : {BRIDGE_STATUS}")
print(f"NEXT_ACTION   : {NEXT_ACTION}")


# ── 원본 불변성 확인 ─────────────────────────────────────────
immut_rows = []
IMMUT_PASS = True

for abs_path, before in immut_records.items():
    p = Path(abs_path)
    exists = p.exists()
    if not exists:
        IMMUT_PASS = False
        immut_rows.append({"absolute_path": abs_path,
                           "label": before["label"],
                           "unchanged": False, "status": "MISSING"})
        continue
    st   = p.stat()
    sha  = sha256_file(p)
    ok   = (int(st.st_size) == before["size_before"]
            and sha == before["sha256_before"])
    if not ok:
        IMMUT_PASS = False
    immut_rows.append({
        "absolute_path":   abs_path,
        "label":           before["label"],
        "size_before":     before["size_before"],
        "size_after":      int(st.st_size),
        "sha256_before":   before["sha256_before"],
        "sha256_after":    sha,
        "unchanged":       ok,
        "status":          "UNCHANGED" if ok else "MODIFIED",
    })

if not IMMUT_PASS:
    BRIDGE_STATUS = "BRIDGE_INVALID_SOURCE_MODIFIED"
    NEXT_ACTION   = "MANUAL_REVIEW_REQUIRED"

IMMUT_DF = pd.DataFrame(immut_rows)
IMMUT_DF.to_csv(OUTPUT_DIR / "source_immutability_check.csv",
                index=False, encoding="utf-8-sig")

print(f"\n[IMMUT] source_files_unchanged = {IMMUT_PASS}")


# ── bridge_spec_candidate.json ───────────────────────────────
spec_cand = {
    "bridge_version":                       "M0_BRIDGE_2048_v1",
    "status":                               BRIDGE_STATUS,
    "selected_bridge":                      "DURATION_PRESERVING_2048" if B_TOTAL_PASS else "UNDECIDED",
    "source_dataset":                       "FEMTO",
    "source_data_type":                     "VIBRATION_ONLY",
    "auxiliary_temperature_excluded":        True,
    "source_fs_hz":                         SOURCE_FS,
    "source_sample_count":                  SOURCE_SAMPLES,
    "source_duration_sec":                  SOURCE_DUR,
    "resample_up":                          RESAMPLE_UP,
    "resample_down":                        RESAMPLE_DOWN,
    "conversion_ratio":                     RESAMPLE_UP / RESAMPLE_DOWN,
    "output_fs_hz":                         OUTPUT_FS,
    "output_sample_count":                  OUTPUT_SAMPLES,
    "output_duration_sec":                  OUTPUT_SAMPLES / OUTPUT_FS,
    "common_analysis_band_hz":              list(COMMON_BAND_HZ),
    "axis":                                 "H",
    "column_index":                         COL_H,
    "selection_dataset":                    "LEARNING_ONLY",
    "selection_sample_count":               TOTAL_SAMPLES,
    "test_used_for_selection":              False,
    "full_test_used_for_selection":         False,
    "no_low_rate_dataset_upsampling":        True,
    "output_information_length_not_increased": True,
    "original_m0_modified":                 False,
    "baseline_frozen_json_modified":        False,
    "source_immutability_pass":             IMMUT_PASS,
    "struct_pass_detail":                   struct_pass,
    "quality_metrics":                      b_metrics,
    "quality_pass_detail":                  quality_pass,
}
write_json(OUTPUT_DIR / "bridge_spec_candidate.json", spec_cand)

# ── bridge_spec_locked.json (PASS 시만) ─────────────────────
if BRIDGE_STATUS == "BRIDGE_SPEC_READY":
    spec_locked = {
        "bridge_version":                       "M0_BRIDGE_2048_v1",
        "status":                               "BRIDGE_SPEC_READY",
        "selected_bridge":                      "DURATION_PRESERVING_2048",
        "source_dataset":                       "FEMTO",
        "source_data_type":                     "VIBRATION_ONLY",
        "auxiliary_temperature_excluded":        True,
        "source_fs_hz":                         SOURCE_FS,
        "source_sample_count":                  SOURCE_SAMPLES,
        "source_duration_sec":                  SOURCE_DUR,
        "resample_up":                          RESAMPLE_UP,
        "resample_down":                        RESAMPLE_DOWN,
        "conversion_ratio":                     RESAMPLE_UP / RESAMPLE_DOWN,
        "output_fs_hz":                         OUTPUT_FS,
        "output_sample_count":                  OUTPUT_SAMPLES,
        "output_duration_sec":                  OUTPUT_SAMPLES / OUTPUT_FS,
        "common_analysis_band_hz":              list(COMMON_BAND_HZ),
        "axis":                                 "H",
        "column_index":                         COL_H,
        "selection_dataset":                    "LEARNING_ONLY",
        "selection_sample_count":               TOTAL_SAMPLES,
        "test_used_for_selection":              False,
        "full_test_used_for_selection":         False,
        "no_low_rate_dataset_upsampling":        True,
        "output_information_length_not_increased": True,
        "original_m0_modified":                 False,
        "baseline_frozen_json_modified":        False,
        "source_immutability_pass":             True,
    }
    write_json(OUTPUT_DIR / "bridge_spec_locked.json", spec_locked)
    print("✅ bridge_spec_locked.json 생성 완료")
else:
    print(f"⚠️  bridge_spec_locked.json 생성 안 함 (status={BRIDGE_STATUS})")


# ── bridge_candidate_summary.csv ─────────────────────────────
summary_rows = []
for cand_name, df_cand, ref_to_orig in [
    ("CENTER_CROP_2048", CAND_A_DF, "crop_vs_crop"),
    ("DURATION_PRESERVING_2048", CAND_B_DF, "full_signal"),
]:
    pass_df = df_cand[df_cand.get("status", pd.Series("ERROR")) == "PASS"] \
              if "status" in df_cand.columns else df_cand
    for metric in ["rms_relerr", "energy_0_6khz_relerr", "dominant_freq_abserr"]:
        if metric in pass_df.columns:
            v = pass_df[metric].dropna()
            summary_rows.append({
                "candidate": cand_name,
                "metric": metric,
                "n": len(v),
                "median": float(v.median()) if len(v)>0 else None,
                "p95": float(np.percentile(v, 95)) if len(v)>0 else None,
                "max": float(v.max()) if len(v)>0 else None,
                "reference": ref_to_orig,
            })

SUMMARY_DF = pd.DataFrame(summary_rows)
SUMMARY_DF.to_csv(OUTPUT_DIR / "bridge_candidate_summary.csv",
                  index=False, encoding="utf-8-sig")


# ── FINAL_BRIDGE_PREP_SUMMARY ─────────────────────────────────
final_json = {
    "version":                  VERSION,
    "created_at":               datetime.now().isoformat(timespec="seconds"),
    "bridge_status":            BRIDGE_STATUS,
    "recommended_next_action":  NEXT_ACTION,
    "source_lock_status":       SOURCE_LOCK_STATUS,
    "classification_pass":      classification_pass,
    "learning_vibration_csv":   LEARNING_VIB_EXPECTED,
    "learning_temp_excluded":   LEARNING_TEMP_EXPECTED,
    "sample_count":             len(SAMPLE_DF),
    "candidate_B_struct_pass":  B_STRUCT_PASS,
    "candidate_B_quality_pass": B_QUALITY_PASS,
    "candidate_B_total_pass":   B_TOTAL_PASS,
    "quality_metrics":          b_metrics,
    "quality_criteria":         BRIDGE_QUALITY_CRITERIA,
    "quality_pass_detail":      quality_pass,
    "struct_pass_detail":       struct_pass,
    "failure_count":            len(FAIL_DF),
    "source_files_unchanged":   IMMUT_PASS,
    "original_files_modified":  False,
    "bridge_conversion_applied_to_full_dataset": False,
    "output_directory":         str(OUTPUT_DIR),
}
write_json(OUTPUT_DIR / "FINAL_BRIDGE_PREP_SUMMARY.json", final_json)

# 품질 기준 표
_q_lines = "\n".join(
    f"| {k} | {v:.4f} | {BRIDGE_QUALITY_CRITERIA[k]} | {'✅ PASS' if quality_pass[k] else '❌ FAIL'} |"
    for k, v in b_metrics.items() if v is not None
)

summary_md = f"""# M0_BRIDGE_2048_PREP_v1.1 Final Summary

## 최종 결과

| 항목 | 값 |
|:---|:---|
| **BRIDGE STATUS** | `{BRIDGE_STATUS}` |
| **NEXT ACTION** | `{NEXT_ACTION}` |
| Source Lock | `{SOURCE_LOCK_STATUS}` |
| Classification pass | `{classification_pass}` |
| Learning vibration CSV | `{LEARNING_VIB_EXPECTED:,}` |
| Learning temp excluded | `{LEARNING_TEMP_EXPECTED:,}` |
| Sample count | `{len(SAMPLE_DF)}` |
| Failures | `{len(FAIL_DF)}` |
| Source unchanged | `{IMMUT_PASS}` |

## Candidate B 품질 기준 평가

| 지표 | 실제값 | 기준 | 결과 |
|:---|---:|---:|:---:|
{_q_lines}

## Candidate 비교

| 후보 | 출력 Fs | 지속시간 | 원본 커버리지 | 비교 기준 |
|:---|---:|---:|---:|:---|
| A: CENTER_CROP_2048 | 25,600 Hz | 0.08 s | 80% | 동일 구간 crop vs crop |
| B: DURATION_PRESERVING_2048 | 20,480 Hz | 0.1 s | 100% | 전체 신호 vs 전체 신호 |

## Source Lock 정정 판정

- 850개 `FAIL_UNRESOLVED_SCHEMA` = `temp_*.csv` 온도 보조 파일 (손상 아님)
- Bridge 입력에서 제외됨 (삭제·이동 없음)
- `SOURCE_LOCK_PASS_WITH_AUXILIARY_FILES_EXCLUDED` 로 정정
"""

with (OUTPUT_DIR / "FINAL_BRIDGE_PREP_SUMMARY.md").open("w", encoding="utf-8") as f:
    f.write(summary_md)

print("\n" + "=" * 72)
print(f"BRIDGE STATUS  : {BRIDGE_STATUS}")
print(f"NEXT ACTION    : {NEXT_ACTION}")
print(f"Struct PASS    : {B_STRUCT_PASS}")
print(f"Quality PASS   : {B_QUALITY_PASS}")
print(f"Failures       : {len(FAIL_DF)}")
print(f"Source immut.  : {IMMUT_PASS}")
print(f"OUTPUT_DIR     : {OUTPUT_DIR}")
print("=" * 72)


# ── 필수 출력 파일 검증 ───────────────────────────────────────
REQUIRED_OUTPUTS = [
    "source_classification_closure.json",
    "source_classification_manifest.csv",
    "auxiliary_temperature_inventory.csv",
    "bridge_sample_manifest.csv",
    "bridge_candidate_A_metrics.csv",
    "bridge_candidate_B_metrics.csv",
    "bridge_candidate_summary.csv",
    "bridge_failures.csv",
    "bridge_spec_candidate.json",
    "source_immutability_check.csv",
    "FINAL_BRIDGE_PREP_SUMMARY.md",
    "FINAL_BRIDGE_PREP_SUMMARY.json",
]

if BRIDGE_STATUS == "BRIDGE_SPEC_READY":
    REQUIRED_OUTPUTS.append("bridge_spec_locked.json")

missing = [n for n in REQUIRED_OUTPUTS if not (OUTPUT_DIR / n).exists()]
if missing:
    print(f"⚠️  누락 파일: {missing}")
else:
    print("✅ 모든 필수 출력 파일 생성 완료")

=== Candidate B 구조 기준 ===
  ✅ all_180_output_2048: True
  ✅ no_nan_inf: True
  ✅ duration_0_1_sec: True
  ✅ effective_fs_20480: True
  ✅ no_low_rate_upsampling: True
  ✅ info_len_not_increased: True
  ✅ bearing_phase_failure_0: True

=== Candidate B 품질 기준 ===
  ✅ energy_0_6khz_relerr_median: 0.0012 <= 0.05  → True
  ✅ energy_0_6khz_relerr_p95: 0.0017 <= 0.1  → True
  ✅ dominant_freq_abserr_median: 0.0000 <= 50.0  → True
  ✅ rms_relerr_median: 0.0408 <= 0.05  → True

BRIDGE_STATUS : BRIDGE_SPEC_READY
NEXT_ACTION   : READY_FOR_M0_BRIDGE_2048_APPLY

[IMMUT] source_files_unchanged = True
✅ bridge_spec_locked.json 생성 완료

BRIDGE STATUS  : BRIDGE_SPEC_READY
NEXT ACTION    : READY_FOR_M0_BRIDGE_2048_APPLY
Struct PASS    : True
Quality PASS   : True
Failures       : 0
Source immut.  : True
OUTPUT_DIR     : /content/drive/MyDrive/Colab Notebooks/field_iis3dwb/bridge_outputs/20260725_192744_bridge_prep_v1_1
✅ 모든 필수 출력 파일 생성 완료


In [16]:
import json
import pandas as pd
from pathlib import Path

_pr_candidates = [
    Path("/content/drive/MyDrive/Colab Notebooks/field_iis3dwb"),
    Path("/content/drive/My Drive/Colab Notebooks/field_iis3dwb"),
]
_pr = None
for _c in _pr_candidates:
    if _c.exists():
        _pr = _c
        break

if _pr is None:
    print("PROJECT_ROOT 없음 — 드라이브 마운트 후 실행")
else:
    prep_dirs = sorted(
        [p for p in (_pr / "bridge_outputs").glob("*_bridge_prep_v1_1")
         if p.is_dir() and (p / "FINAL_BRIDGE_PREP_SUMMARY.json").exists()],
        key=lambda p: p.stat().st_mtime_ns, reverse=True,
    )

    if not prep_dirs:
        print("FINAL_BRIDGE_PREP_SUMMARY.json 없음 — 셀 02~05를 먼저 실행하세요")
    else:
        _sd = prep_dirs[0]
        with (_sd / "FINAL_BRIDGE_PREP_SUMMARY.json").open("r", encoding="utf-8") as f:
            s = json.load(f)

        print("=" * 70)
        print(f"BRIDGE STATUS        : {s.get('bridge_status')}")
        print(f"NEXT ACTION          : {s.get('recommended_next_action')}")
        print(f"Source lock          : {s.get('source_lock_status')}")
        print(f"Classification pass  : {s.get('classification_pass')}")
        print(f"Vib CSV (Learning)   : {s.get('learning_vibration_csv')}")
        print(f"Temp excluded        : {s.get('learning_temp_excluded')}")
        print(f"Sample count         : {s.get('sample_count')}")
        print(f"Struct PASS          : {s.get('candidate_B_struct_pass')}")
        print(f"Quality PASS         : {s.get('candidate_B_quality_pass')}")
        print(f"Failures             : {s.get('failure_count')}")
        print(f"Source unchanged     : {s.get('source_files_unchanged')}")
        print(f"Output dir           : {s.get('output_directory')}")
        print("=" * 70)

        print("\n품질 지표:")
        qm = s.get("quality_metrics", {})
        qp = s.get("quality_pass_detail", {})
        qc = s.get("quality_criteria", {})
        for k, v in qm.items():
            thr = qc.get(k, "?")
            ok  = qp.get(k, False)
            icon = "✅" if ok else "❌"
            print(f"  {icon} {k}: {v} (기준 ≤ {thr})")

        # 실패 파일 표시
        fail_path = _sd / "bridge_failures.csv"
        if fail_path.exists():
            try:
                # pandas can raise EmptyDataError even if file exists but has no columns/data
                fdf = pd.read_csv(fail_path)
                if len(fdf) > 0:
                    print(f"\n⚠️  Bridge 실패 파일 ({len(fdf)}개):")
                    display(fdf)
                else:
                    # File exists, but read as empty DataFrame (e.g., only header, or truly empty after read)
                    print("\n✅ Bridge 실패 파일: 없음")
            except pd.errors.EmptyDataError:
                # Handle case where file exists but pandas can't parse any data/columns (e.g., completely empty file)
                print("\n✅ Bridge 실패 파일: 없음")
        else:
            # File does not exist
            print("\n✅ Bridge 실패 파일: 없음")

        # spec_locked 확인
        locked_path = _sd / "bridge_spec_locked.json"
        if locked_path.exists():
            with locked_path.open("r", encoding="utf-8") as f:
                locked = json.load(f)
            print("\n🔒 bridge_spec_locked.json:")
            print(json.dumps(locked, ensure_ascii=False, indent=2))

BRIDGE STATUS        : BRIDGE_SPEC_READY
NEXT ACTION          : READY_FOR_M0_BRIDGE_2048_APPLY
Source lock          : SOURCE_LOCK_PASS_WITH_AUXILIARY_FILES_EXCLUDED
Classification pass  : True
Vib CSV (Learning)   : 7534
Temp excluded        : 850
Sample count         : 180
Struct PASS          : True
Quality PASS         : True
Failures             : 0
Source unchanged     : True
Output dir           : /content/drive/MyDrive/Colab Notebooks/field_iis3dwb/bridge_outputs/20260725_192744_bridge_prep_v1_1

품질 지표:
  ✅ energy_0_6khz_relerr_median: 0.0012461036056635568 (기준 ≤ 0.05)
  ✅ energy_0_6khz_relerr_p95: 0.0016738085000070975 (기준 ≤ 0.1)
  ✅ dominant_freq_abserr_median: 0.0 (기준 ≤ 50.0)
  ✅ rms_relerr_median: 0.04079382433787834 (기준 ≤ 0.05)

✅ Bridge 실패 파일: 없음

🔒 bridge_spec_locked.json:
{
  "bridge_version": "M0_BRIDGE_2048_v1",
  "status": "BRIDGE_SPEC_READY",
  "selected_bridge": "DURATION_PRESERVING_2048",
  "source_dataset": "FEMTO",
  "source_data_type": "VIBRATION_ONLY",
  "auxil